# qchem_colab_app(仮) UMA版 — 機械学習ポテンシャルによる構造最適化・振動数解析

Google Colab上で、機械学習ポテンシャル **UMA**(Meta FAIR Chemistry)を使った
構造最適化と振動数解析を行います。DFT等の量子化学計算よりはるかに高速ですが、
**結果は機械学習による予測値であり、量子化学計算そのものの値ではありません。**
(量子化学計算を行いたい場合はこちら: https://colab.research.google.com/github/orgchem610/qchem_colab_app/blob/main/ColabApp.ipynb)

**上から順にセルを実行**していく構成になっています。

**初回の準備(下の「事前準備」: Hugging Faceのトークン)が済んでいれば、メニューの「ランタイム」→**
**「すべてのセルを実行」を押すだけで、最後に操作画面が表示されるまで自動で進みます**
(最初に1回だけ、トークンの入力が必要です。Colabの「シークレット」に保存しておけば、その入力も不要です)。

- **I. Setup Section** — 環境構築(ランタイム設定・コードとライブラリの準備・Hugging Faceの認証)
- **II. Execution Section** — 実際の計算画面(GUI)

初めてJupyter/Colabを使う方へ: 各コードセルの左側にマウスを合わせると再生ボタン(▶)が
表示されます。それをクリックするか、セルを選択して `Shift+Enter` を押すと、そのセルが実行されます。
**必ず上のセルから順番に**実行してください(途中を飛ばすとエラーになります)。

## 注意事項
**本プログラムは研究用に作成されたものではありません**
DFT等の量子化学計算よりはるかに高速ですが、
**結果は機械学習による予測値であり、量子化学計算そのものの値ではありません。**
(量子化学計算を行いたい場合はこちら: https://colab.research.google.com/github/orgchem610/qchem_colab_app/blob/main/ColabApp.ipynb)

## UMAのライセンスと引用について

本プログラムでは、Meta FAIR Chemistry の機械学習ポテンシャル **UMA** を使用します。
UMAの利用・引用にあたっては、以下の一次情報をご確認ください。

- ライセンス(FAIR Chemistry License v1): https://huggingface.co/facebook/UMA/blob/main/LICENSE
- Acceptable Use Policy・モデルカード: https://huggingface.co/facebook/UMA
- fairchem(ソースコード、MITライセンス): https://github.com/facebookresearch/fairchem

## 使い方

### Hugging Face アクセストークンの発行

UMAの学習済みモデルを利用するために、Hugging Faceで利用申請をする必要があります。

1. https://huggingface.co でアカウントを作成してログインします。
2. https://huggingface.co/facebook/UMA を開き、ライセンスと Acceptable Use Policy を確認したうえで、
   フォームに入力して利用条件に同意し、アクセスを申請します(承認まで時間がかかる場合があります)。
3. https://huggingface.co/settings/tokens で、権限に **Read** を含む **アクセストークン**(`hf_` で始まる文字列)を作成します。
(詳細は`ColabReaction`ユーザーガイド(https://bilab.github.io/ColabReaction/User_Guide_JP.pdf)の`Hugging Face Access token の取得方法`の章を参照して下さい)

### アクセストークンの登録

- 画面左端にある鍵のアイコンをクリックし、「シークレット」と書かれたパネルを開いて下さい
- 「新しいシークレットを追加」ボタンを押し、「名前」に`HF_TOKEN`、
  「値」に発行したHugging Face アクセストークン(`hf_...`)を入力してください
- 「ノートブックからのアクセス」の欄をクリックし、チェックが入った状態にして下さい

**トークンは他人に教えたり、ノートブックやGitHubに書き込んだり、コードに直接書いたりしないでください**
(「シークレット」欄に入力した値は外部に公開されません)

## I. Setup Section

### ランタイムの設定

デフォルトでは以下の通り設定されています

| 項目 | 設定 | 理由 |
|---|---|---|
| **ハードウェア アクセラレータ** | **T4 GPU** | UMAはGPUで高速に動作します |
| **ランタイム バージョン** | **2026.07** | Python 3.12 / PyTorch 2.11.0 などを固定し、Google側の更新で動作が変わるのを防ぎます |

- ランタイムの設定を変更したい場合、上部メニューの **「ランタイム」→「ランタイムのタイプを変更」** を開き、設定を変更してください。

- 「ランタイム バージョン」を「最新」とすると起動が1,2分ほど早くなりますが、動作の保証はできません


In [ ]:
#@title 1. 現在の実行環境を確認する
import importlib.metadata as md
import platform
import subprocess

# torchは、「3. 依存ライブラリのインストール」で fairchem-core により 2.11 → 2.13 に更新される
# ことを確認しているため、どちらの版も想定内として扱う。
EXPECTED = {"python": ("3.12",), "numpy": ("2.0",), "torch": ("2.11", "2.13")}

def _version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

print(f"Python : {platform.python_version()}")
print(f"numpy  : {_version('numpy')}")
print(f"torch  : {_version('torch')}")

try:
    gpu_name = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=15).stdout.strip()
except (FileNotFoundError, OSError, subprocess.TimeoutExpired):
    gpu_name = ""
print(f"GPU    : {gpu_name or '(検出されませんでした)'}")

problems = []
if not platform.python_version().startswith(EXPECTED["python"]):
    problems.append(f"Pythonが {EXPECTED['python'][0]}.x ではありません(ランタイムバージョンが 2026.07 でない可能性があります)")
for pkg in ("numpy", "torch"):
    v = _version(pkg)
    if v is None or not v.startswith(EXPECTED[pkg]):
        problems.append(f"{pkg} が {' または '.join(EXPECTED[pkg])}.x ではありません")
if "T4" not in gpu_name:
    problems.append("T4 GPUが検出されませんでした(GPUなしではUMAの計算に非常に時間がかかります)")

if problems:
    print("\n⚠ 次の点が想定と異なります:")
    for p in problems:
        print("  -", p)
    print("「ランタイム」→「ランタイムのタイプを変更」で設定を確認してください。"
          "設定を変えるとセッションが再接続されるため、このセルから実行し直してください。")
else:
    print("\n✓ 想定どおりの環境です(ランタイム 2026.07 / T4 GPU)。")
print("(torchは、「3.」のインストール後に 2.11.0 → 2.13.0 に更新されます)")

# 以降のインストールで torch / numpy が入れ替わったかを調べるための記録
VERSIONS_BEFORE_INSTALL = {"numpy": _version("numpy"), "torch": _version("torch"),
                           "ipywidgets": _version("ipywidgets")}

### コード本体の取得

このプロジェクトのコードは https://github.com/orgchem610/qchem_colab_app で管理しています。
デフォルトでは`main`ブランチの最新版を取得して実行します。

ブランチやバージョンを指定する場合は、 https://github.com/orgchem610/qchem_colab_app/branches や
https://github.com/orgchem610/qchem_colab_app/tags からブランチ名やタグ名を選択し、
「1. コードを取得する」セルの`BRANCH`欄に入力してから、「すべてのセルを実行」を押して下さい


In [ ]:
#@title 1-a. コードを取得する
import os
import re
import subprocess
import sys

BRANCH = "main"  #@param ["main", "develop"] {allow-input: true}

GITHUB_USER = "orgchem610"
GITHUB_REPO = "qchem_colab_app"
REPO_URL = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
PROJECT_DIR = f"/content/{GITHUB_REPO}"

BRANCH = BRANCH.strip()
if not re.fullmatch(r"[A-Za-z0-9._/-]+", BRANCH) or BRANCH.startswith("-"):
    raise ValueError(f"ブランチ名(タグ名)として使えない文字が含まれています: {BRANCH!r}")


def run_git(*args, cwd=None):
    """gitコマンドを実行し、(成功したか, 標準出力, 標準エラー) を返す。"""
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    return r.returncode == 0, r.stdout.strip(), r.stderr.strip()


def fail(message, detail=""):
    print(f"❌ {message}")
    if detail:
        print(detail[-600:])
    print("ブランチ名・タグ名の綴りをご確認ください。")
    raise RuntimeError(message)


previous_head = None
if not os.path.exists(PROJECT_DIR):
    print(f"{GITHUB_REPO} をクローンします(取得するブランチ/タグ: {BRANCH})...")
    ok, _, err = run_git("clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR)
    if not ok:
        fail("git clone に失敗しました。", err)
else:
    print(f"{PROJECT_DIR} は既に存在するため、{BRANCH} に切り替えて最新版を取得します...")
    _, previous_head, _ = run_git("rev-parse", "HEAD", cwd=PROJECT_DIR)
    ok, _, err = run_git("fetch", "--tags", "origin", cwd=PROJECT_DIR)
    if not ok:
        fail("GitHubからの情報取得(git fetch)に失敗しました。", err)
    ok, _, err = run_git("checkout", BRANCH, cwd=PROJECT_DIR)
    if not ok:
        fail(f"{BRANCH} への切り替え(git checkout)に失敗しました。", err)
    # ブランチなら最新の内容を取り込む。タグは内容が固定されているため、取り込みは不要。
    on_branch, _, _ = run_git("symbolic-ref", "-q", "HEAD", cwd=PROJECT_DIR)
    if on_branch:
        ok, _, err = run_git("pull", "--ff-only", cwd=PROJECT_DIR)
        if not ok:
            fail("最新版の取得(git pull)に失敗しました。", err)

_, head, _ = run_git("rev-parse", "HEAD", cwd=PROJECT_DIR)
_, current_branch, _ = run_git("branch", "--show-current", cwd=PROJECT_DIR)
_, last_commit, _ = run_git(
    "log", "-1", "--format=%h %s (%cd)", "--date=short", cwd=PROJECT_DIR)
_, described, _ = run_git("describe", "--tags", "--always", cwd=PROJECT_DIR)

print("\n=== 取得したコード ===")
if current_branch:
    print(f"  ブランチ        : {current_branch}")
else:
    print(f"  タグ(固定)    : {BRANCH}")
print(f"  最新コミット    : {last_commit}")
print(f"  git describe    : {described}")

if previous_head and previous_head != head and any(
        name in sys.modules for name in ("qchem_colab_app",)):
    print(
        "\n⚠ 取得したコードが前回から変わりましたが、Pythonは古いコードを読み込んだままです。\n"
        "  「ランタイム」→「セッションを再起動」を実行し、このセルから上から順に実行し直してください。")

In [ ]:
#@title 1-b. 取得したコードの場所をPythonに登録する
import sys
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)
print("sys.pathに追加しました:", PROJECT_DIR)

### 2. Hugging Faceのトークンの入力とUMAへのアクセスの確認

登録したトークンでHugging Faceにログインし、UMAにアクセスできるかを確認します
# トークンを入力し直したい場合は、`FORCE_REENTER`にチェックを入れてから実行してください。


In [ ]:
#@title 2. UMAへのアクセスを確認する
FORCE_REENTER = False  #@param {type:"boolean"}

import os
from qchem_colab_app.uma_backend import uma_hf

# --- 1. トークンの受け取り ---------------------------------------------------
# 受け渡しは、Hugging Face公式の標準の環境変数 HF_TOKEN だけで行い、変数には残さない。
_token = None
if os.environ.get("HF_TOKEN") and not FORCE_REENTER:
    print("✓ Hugging Faceのトークンは、このセッションで設定済みです。入力は不要です。")
else:
    try:
        from google.colab import userdata
        _token = userdata.get("HF_TOKEN")     # Colabのシークレットに保存してある場合
    except Exception:
        _token = None                         # シークレット未設定・未許可の場合は、下の入力欄を使う

    if _token:
        print("Colabのシークレット(HF_TOKEN)を使用します。")
    else:
        from getpass import getpass
        _token = getpass("Hugging Faceのアクセストークン(hf_ で始まる文字列)を入力して、"
                         "Enterキーを押してください(入力内容は表示されません): ").strip()

    if _token:
        os.environ["HF_TOKEN"] = _token
        print("✓ トークンを受け付けました。")
_token = None                                 # 変数にトークンを残さない

# --- 2. Hugging Faceへのログインと、UMAへのアクセスの確認 --------------------------
try:
    uma_hf.login_from_environment()
    uma_hf.check_model_access()
    print("✓ Hugging Faceにログインし、UMAへのアクセスを確認しました。以降のセルは自動で進みます。")
except uma_hf.UmaSetupError as e:
    print("❌ UMAを使うための準備が完了していません:")
    print(e)
    print("\n原因を解消したら、このセルを実行し直してください。トークンを入力し直す場合は、"
          "右上の「FORCE_REENTER」にチェックを入れてから実行してください。")

### 依存ライブラリのインストール

`プログラムの実行に必要なライブラリをインストールします。2分くらいかかります。


In [ ]:
#@title 3-a. 依存ライブラリをインストールする
import importlib.metadata as md
import os
import subprocess
import time

def _version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

# 「1. 現在の実行環境を確認する」を実行済みならその記録を、(セッション再起動後などで)
# 無ければ現在の版を、「インストール前の版」として扱う。
VERSIONS_BEFORE_INSTALL = globals().get("VERSIONS_BEFORE_INSTALL") or {
    "numpy": _version("numpy"), "torch": _version("torch"), "ipywidgets": _version("ipywidgets")}

req_lock = os.path.join(PROJECT_DIR, "requirements-uma-lock.txt")
req_plain = os.path.join(PROJECT_DIR, "requirements-uma.txt")
req_file = req_lock if os.path.exists(req_lock) else req_plain
print(f"インストールに使用するファイル: {req_file}")

with open(req_file) as f:
    lines = [ln.strip() for ln in f if ln.strip() and not ln.strip().startswith("#")]

failed = []
t0 = time.time()
for pkg in lines:
    print(f"--- installing: {pkg} ---")
    result = subprocess.run(["pip", "install", "-q", pkg], capture_output=True, text=True)
    if result.returncode != 0:
        failed.append(pkg)
        print(f"❌ {pkg} のインストールに失敗しました。エラー末尾:")
        print("\n".join(result.stderr.strip().splitlines()[-15:]))
    else:
        print(f"OK: {pkg}")
print(f"\nインストール所要時間: {time.time() - t0:.0f} 秒")

if failed:
    print("\n=== 以下のパッケージのインストールに失敗しました ===")
    for pkg in failed:
        print(" -", pkg)
    print(
        "\n上のエラー内容(特に最後の数行)を確認してください。\n"
        "ランタイムバージョンが 2026.07 でない場合は、torch/numpy/Pythonの版が合わず\n"
        "失敗することがあります(上の「1. 現在の実行環境を確認する」を再確認してください)。\n"
    )
else:
    print("\n全てのパッケージのインストールに成功しました。")

# torch / numpy の版が入れ替わっていないか確認する
import sys

changed = []
for pkg, before in VERSIONS_BEFORE_INSTALL.items():
    after = _version(pkg)
    if before != after:
        changed.append((pkg, before, after))
if changed:
    print("\nインストールにより、次のライブラリの版が変わりました:")
    for pkg, before, after in changed:
        print(f"  - {pkg}: {before} → {after}")
    if any(pkg == "ipywidgets" for pkg, _, _ in changed):
        print("\n(ipywidgetsの版が変わりました。8.x になった場合、セッションを再起動すると画面部品が"
              "表示されなくなることがあります。その場合は、Execution Sectionの「5.」のセルを実行してください。)")
    already_loaded = [pkg for pkg, _, _ in changed if pkg in sys.modules and pkg != "ipywidgets"]
    if already_loaded:
        print(f"\n⚠ {', '.join(already_loaded)} は既にPythonに読み込まれているため、"
              "古い版のまま動いてしまいます。")
        print("「ランタイム」→「セッションを再起動」を実行してから、すべてのセルを実行し直してください。")
    elif any(pkg != "ipywidgets" for pkg, _, _ in changed):
        print("(セッションの再起動は不要です。次のセルへ進んでください。)")

In [ ]:
#@title 3-b. インストールされた版を requirements-uma-lock.txt に記録する
# Colabの pip freeze はプリインストール済みの大量のパッケージも含むため、
# requirements-uma.txt に明示的に書かれているパッケージだけに絞って書き出します。
import re
import subprocess

freeze_result = subprocess.run(["pip", "freeze"], capture_output=True, text=True)
all_lines = freeze_result.stdout.splitlines()

with open(os.path.join(PROJECT_DIR, "requirements-uma.txt")) as f:
    wanted_names = set()
    for line in f:
        line = line.strip()
        if line and not line.startswith("#"):
            wanted_names.add(re.split(r"[<>=!~\[]", line)[0].strip().lower().replace("_", "-"))

def _pkg_name(line):
    return re.split(r"[<>=!~\[ @]", line)[0].strip().lower().replace("_", "-")

filtered_lines = [line for line in all_lines if _pkg_name(line) in wanted_names]

lock_path = os.path.join(PROJECT_DIR, "requirements-uma-lock.txt")
with open(lock_path, "w") as f:
    f.write("\n".join(sorted(filtered_lines)) + "\n")

print(f"requirements-uma-lock.txt を書き出しました({len(filtered_lines)}パッケージ分)。")
for line in sorted(filtered_lines):
    print(" -", line)


In [ ]:
#@title 3-c. torchとGPUが使えることを確認する
# fairchem-core のインストールで torch が更新されるため(2.11.0 → 2.13.0)、
# 更新後のtorchがこのランタイムのGPUを使えることを確認します。
try:
    import torch
except ImportError as e:
    print("❌ torch を読み込めませんでした。「3-a. 依存ライブラリのインストール」を確認してください。")
    print(e)
else:
    print(f"torch : {torch.__version__}  (CUDA {torch.version.cuda})")
    if torch.cuda.is_available():
        print(f"GPU   : {torch.cuda.get_device_name(0)}")
        try:
            x = torch.randn(512, 512, device="cuda")
            float((x @ x).sum().item())
            print("\n✓ GPU上での計算に成功しました。")
        except Exception as e:
            print("\n❌ GPU上での計算に失敗しました:")
            print(e)
    else:
        print("\n⚠ torchからGPUを利用できません。このままではUMAの計算が非常に遅くなります。")
        print("  ・「ランタイム」→「ランタイムのタイプを変更」で、T4 GPUが選ばれているか確認してください。")


### 動作確認用サンプル構造の作成

テスト用のサンプル(エタノール, 9原子)を作成します。

In [ ]:
#@title 4. 動作確認用のサンプル構造を作成する
sample_xyz = """9
Ethanol (CH3CH2OH) - rough starting geometry for connectivity testing
C   0.000000   0.000000   0.000000
C   1.510000   0.000000   0.000000
O   1.970000   1.340000   0.000000
H  -0.363000   0.000000   1.028000
H  -0.363000  -0.890000  -0.514000
H  -0.363000   0.890000  -0.514000
H   2.020000  -0.360000   0.890000
H   2.020000  -0.360000  -0.890000
H   1.445000   2.144000   0.000000
"""
with open("/content/sample_ethanol.xyz", "w") as f:
    f.write(sample_xyz)
print("/content/sample_ethanol.xyz を作成しました。")
print("画面左側のフォルダアイコンをクリック→「sample_ethanol.xyz」を右クリック→「ダウンロード」で")
print("サンプルファイルをダウンロードできます")


## II. Execution Section

以下のセルを実行すると操作画面が表示されます。

- 「① 構造ファイルと計算条件」で、構造ファイル(`.xyz` / `.com` / `.gjf` / `.pdb`)を枠に
  ドラッグ＆ドロップ、または枠をクリックして選択し、UMAモデル・電荷・スピン多重度を設定
- 「② 実行」の **「計算を実行する」** ボタンを押す(初回はモデルのダウンロードが伴うため、時間がかかります)
- 「③ 可視化」でエネルギー推移・振動アニメーションを確認
- 「④ ダウンロード」から最終構造(.xyz)・構造推移・計算ログを取得
  (計算ログはGaussViewなどで開けるようなものではなく、プレーンな.txtファイルです)


In [ ]:
#@title 5. ウィジェットを表示する準備
# Colab標準のウィジェット表示の仕組みは ipywidgets 7.x 向けです。ipywidgets 8.x が
# 読み込まれている場合は、「カスタムウィジェットマネージャー」を有効にしないと、
# 画面部品が表示されず「コードスニペット」のパネルが開くだけになります。
import ipywidgets

major = int(ipywidgets.__version__.split(".")[0])
print(f"ipywidgets: {ipywidgets.__version__}")
if major >= 8:
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
        print("✓ カスタムウィジェットマネージャーを有効にしました。次のセルを実行してください。")
    except ImportError:
        print("(Google Colab以外の環境のため、この設定は不要です。)")
else:
    print("Colab標準の仕組みで表示できるため、この設定は不要です。次のセルへ進んでください。")

In [ ]:
#@title 6. 操作画面を表示する
from qchem_colab_app.gui import uma_screen
from IPython.display import display

app = uma_screen.build_app()
display(app)